# M8 Validation: SmolVLA VLA + LIBERO wiring (Linux/Colab T4)

Validates the M8 SmolVLA VLA backend on a Google Colab Linux+CUDA VM (T4 16GB). Covers `.vla-venv` setup, `VLA_POLICY=smolvla` server health, `/predict` wiring smoke, and optional short LIBERO arm rollout.

**Run from Cursor IDE** (default): connect a Colab GPU runtime, open this notebook, edit `GITHUB_BRANCH` in section 1, then run. There is no Colab Secrets UI in Cursor — see SRD § Validation tooling (Colab).

**Branch:** use `main` when M8 is already merged; use a feature branch only when validating pre-merge M8 work.

## 1 — Secrets & clone

**Cursor IDE (default):** edit `GITHUB_BRANCH` in the cell below, then run. `GITHUB_TOKEN` is prompted via `getpass`.

**colab.google.com:** optional Colab Secrets for `GITHUB_BRANCH` / `GITHUB_TOKEN`, or use the same inline settings.

Fresh clone and re-run paths both fetch, check out, and hard-reset to `origin/<GITHUB_BRANCH>`.

In [ ]:
import getpass, os, subprocess, sys

# Cursor IDE: edit here (no Colab Secrets UI). Use a feature branch only for pre-merge M8.
GITHUB_BRANCH = 'main'
os.environ.setdefault('GITHUB_BRANCH', GITHUB_BRANCH)

github_repo = os.environ.get('GITHUB_REPO', 'pmish11/Longor-Robotics')
github_branch = os.environ.get('GITHUB_BRANCH', '').strip()
if not github_branch:
    raise RuntimeError(
        'Set GITHUB_BRANCH in this cell (or Colab Secrets) before running. '
        'Do not validate pre-merge M8 against origin/main.'
    )
print(f'Using branch: {github_branch}')

if not os.environ.get('GITHUB_TOKEN'):
    os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT (repo scope): ')

os.chdir('/content/Longor' if os.path.exists('/content/Longor') else '/')
if '/content/Longor/src' not in sys.path:
    sys.path.insert(0, '/content/Longor/src')

if not os.path.exists('/content/Longor/.git'):
    result = subprocess.run(
        ['git', 'clone',
         f"https://{os.environ['GITHUB_TOKEN']}@github.com/{github_repo}.git",
         '/content/Longor'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stderr.replace(os.environ['GITHUB_TOKEN'], '***'))
        raise RuntimeError('git clone failed')
    print('Cloned OK')

os.chdir('/content/Longor')
if '/content/Longor/src' not in sys.path:
    sys.path.insert(0, '/content/Longor/src')

for cmd in (
    ['git', 'fetch', 'origin', github_branch],
    ['git', 'checkout', github_branch],
    ['git', 'reset', '--hard', f'origin/{github_branch}'],
):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/Longor')
    print(result.stdout or result.stderr or 'ok')
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'{" ".join(cmd)} failed')

## 2 — App deps + LIBERO + headless

In [ ]:
import importlib.util
import logging
import os
import subprocess
import sys

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r requirements-libero.txt

spec = importlib.util.find_spec('robosuite')
if spec:
    script = spec.submodule_search_locations[0] + '/scripts/setup_macros.py'
    result = subprocess.run([sys.executable, script], check=False)
    if result.returncode != 0:
        logging.warning('robosuite setup_macros.py failed (rc=%s); continuing', result.returncode)

## 3 — SmolVLA VLA venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-smolvla-linux.txt')
print('SmolVLA VLA venv ready.')

## 4 — Env config (M8: VLA_POLICY=smolvla)

In [ ]:
import importlib.util, os, yaml

env_content = """VLA_POLICY=smolvla
SMOLVLA_CHECKPOINT=HuggingFaceVLA/smolvla_libero
BACKEND=libero
PERCEPTION_BACKEND=sim_gt
VLA_PORT=8765
VLA_GRIPPER_MODE=absolute_open_close
VLA_ACTION_SCALE=1.0
VLA_STEPS_PER_ACTION=1
VLA_ADAPTIVE_SCALE=false
VLA_CHUNK_STEPS=5
VLA_MAX_INNER_STEPS=50
LIBERO_SETTLE_STEPS=10
LIBERO_EE_DELTA_SCALE=1.0
"""
with open('/content/Longor/.env', 'w') as f:
    f.write(env_content)
print('Wrote /content/Longor/.env (VLA_POLICY=smolvla)')

libero_config_path = os.environ.get('LIBERO_CONFIG_PATH', os.path.expanduser('~/.libero'))
config_file = os.path.join(libero_config_path, 'config.yaml')
if not os.path.exists(config_file):
    os.makedirs(libero_config_path, exist_ok=True)
    spec = importlib.util.find_spec('libero')
    libero_pkg = os.path.join(os.path.dirname(spec.origin), 'libero')
    default_config = {
        'benchmark_root': libero_pkg,
        'bddl_files': os.path.join(libero_pkg, 'bddl_files'),
        'init_states': os.path.join(libero_pkg, 'init_files'),
        'datasets': os.path.join(os.path.dirname(libero_pkg), 'datasets'),
        'assets': os.path.join(libero_pkg, 'assets'),
    }
    with open(config_file, 'w') as f:
        yaml.dump(default_config, f)
    print(f'Created {config_file}')
else:
    print(f'Already exists: {config_file}')

## 5 — Start SmolVLA VLA server + health

In [ ]:
import json, os, subprocess, time, urllib.request

VLA_LOG = '/tmp/m8_vla_smolvla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'

def _tail_log(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'

subprocess.run(['fuser', '-k', '8765/tcp'], capture_output=True)

log = open(VLA_LOG, 'w', buffering=1)
env_for_server = {
    **os.environ,
    'VLA_POLICY': 'smolvla',
    'SMOLVLA_CHECKPOINT': 'HuggingFaceVLA/smolvla_libero',
}
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor',
    env=env_for_server,
)

POLL_SEC = 15
MAX_WAIT_SEC = 15 * 60
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    elapsed = (i + 1) * POLL_SEC
    if vla_proc.poll() is not None:
        print(_tail_log(60))
        raise RuntimeError(f'VLA server exited early (code {vla_proc.returncode}). Log above.')
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        body = json.loads(resp.read())
        print('VLA server ready:', body)
        assert body.get('model') == 'smolvla', body
        break
    except Exception:
        print(f'Waiting... ({elapsed}s)')
        if elapsed % 60 == 0:
            print('--- recent log ---')
            print(_tail_log(15))
else:
    print(_tail_log(60))
    raise RuntimeError(f'VLA server did not come up in {MAX_WAIT_SEC // 60} minutes. Log above.')

## 6 — Wiring smoke: unit tests + live /predict

In [ ]:
import io, json, os, subprocess, sys, urllib.request
from datetime import datetime
from PIL import Image

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m8_smolvla_smoke_{stamp}.txt'

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'services/tests/test_smolvla_service.py', '-v'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'},
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(combined[-4000:])
assert result.returncode == 0, f'smolvla unit tests failed (exit {result.returncode})'

buf = io.BytesIO()
Image.new('RGB', (64, 64), color=(120, 80, 40)).save(buf, format='JPEG')
jpeg = buf.getvalue()

import requests
resp = requests.post(
    'http://localhost:8765/predict',
    files={'image': ('frame.jpg', jpeg, 'image/jpeg')},
    data={'task': 'pick up the cube'},
    timeout=120,
)
print('Live /predict status:', resp.status_code)
body = resp.json()
print('actions rows:', len(body.get('actions', [])))
assert resp.status_code == 200, resp.text
actions = body['actions']
assert isinstance(actions, list) and len(actions) >= 1
assert all(isinstance(row, list) and len(row) == 7 for row in actions)
print('M8 SmolVLA wiring smoke PASS')

## 7 — Optional: short LIBERO arm rollout (sim_gt)

In [ ]:
import os, subprocess, sys
from datetime import datetime

RUN_ARM_SMOKE = os.environ.get('M8_RUN_ARM_SMOKE', '1').strip() not in ('0', 'false', 'no')
if not RUN_ARM_SMOKE:
    print('Skipping arm rollout (M8_RUN_ARM_SMOKE=0)')
else:
    trace_dir = '/content/Longor/traces'
    os.makedirs(trace_dir, exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    capture_path = f'{trace_dir}/m8_arm_smoke_{stamp}.txt'
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', 'tests/test_libero_integration.py', '-m', 'integration', '-k', 'arm', '-v', '--tb=short'],
        cwd='/content/Longor',
        capture_output=True, text=True, timeout=900,
        env={
            **os.environ,
            'PYTHONPATH': '/content/Longor/src',
            'MUJOCO_GL': 'egl',
            'PYOPENGL_PLATFORM': 'egl',
            'VLA_POLICY': 'smolvla',
            'VLA_URL': 'http://localhost:8765',
        },
    )
    combined = result.stdout + result.stderr
    with open(capture_path, 'w') as f:
        f.write(combined)
    print(f'Capture: {capture_path}')
    print(combined[-6000:])
    if result.returncode != 0:
        print('Arm smoke failed or no integration tests matched — check capture (non-fatal for M8 wiring).')
    else:
        print('M8 arm rollout smoke PASS')

## 8 — pytest + teardown

In [ ]:
import os, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m8_pytest_{stamp}.txt'

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'not integration', '--tb=short', '-q'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'},
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(combined[-4000:])
assert result.returncode == 0, f'pytest failed (exit {result.returncode})'

vla_proc.terminate()
print('VLA server stopped. M8 validation complete.')